# 02 · Análisis exploratorio (EDA) y control de duplicados
**Entrega 1 · cronograma días 3–5.** No necesita GPU. Requiere haber ejecutado el notebook 01.

Preguntas que responde: ¿cuántas fotos hay de cada clase?, ¿cómo se ven?, ¿hay fotos repetidas entre
entrenamiento y prueba (fuga de información)?

In [ ]:
# PREPARACIÓN (igual en todos los notebooks): conecta Drive, entra a la carpeta del proyecto y deja listos los datos.
from google.colab import drive
drive.mount('/content/drive')

PROYECTO = '/content/drive/MyDrive/yuca-comparacion'   # carpeta del proyecto dentro de su Google Drive

import os, sys
os.chdir(PROYECTO)
sys.path.insert(0, PROYECTO)

from src.datos import descargar_dataset
# El .zip se guarda una sola vez en Drive; las fotos se descomprimen en el disco de Colab (más rápido).
RAIZ = str(descargar_dataset(destino='/content/datos', carpeta_zip=PROYECTO + '/data'))
os.environ['YUCA_DATOS'] = RAIZ
print('Proyecto:', os.getcwd())
print('Datos   :', RAIZ)

In [ ]:
import pandas as pd
from IPython.display import Image as Mostrar, display
from src import eda
inventario = pd.read_csv('resultados/inventario.csv')
len(inventario)

## 1. Distribución de clases
Si una clase tiene muchas más fotos que otra, hay **desbalance**: un modelo puede acertar mucho solo por preferir la clase grande.

In [ ]:
tabla = eda.grafica_distribucion(inventario)
display(Mostrar('resultados/eda/distribucion_clases.png'))
tabla

## 2. Tamaño de las imágenes

In [ ]:
eda.resumen_resoluciones(inventario)

## 3. Ejemplos de cada clase
Miren las fotos con calma: fondos, iluminación, si la hoja ocupa toda la imagen, si hay manos o suelo. Anoten lo que observen.

In [ ]:
display(Mostrar(str(eda.mosaico_ejemplos(inventario, RAIZ, por_clase=5, semilla=0))))

## 4. Brillo promedio por clase
Si una clase suele estar más clara u oscura, el modelo podría usar el brillo como **atajo** en vez de mirar la lesión.

In [ ]:
eda.brillo_medio(inventario, RAIZ)

## 5. Duplicados exactos
Archivos idénticos byte a byte. Los que tienen `entre_particiones = True` son fuga de información.

In [ ]:
exactos = eda.duplicados_exactos(inventario)
exactos.to_csv('resultados/eda/duplicados_exactos.csv', index=False)
print('Archivos repetidos:', len(exactos), '| grupos:', exactos['md5'].nunique(),
      '| archivos en grupos que cruzan particiones:', int(exactos['entre_particiones'].sum()))
exactos.head(20)

## 6. Casi duplicados
Fotos visualmente casi iguales aunque el archivo sea distinto. Son **candidatos**: hay que mirarlos. Tarda unos minutos.

In [ ]:
casi = eda.casi_duplicados(inventario, RAIZ, distancia_max=2)
casi.to_csv('resultados/eda/casi_duplicados.csv', index=False)
print('Pares candidatos:', len(casi), '| entre particiones:', int(casi['entre_particiones'].sum()))
casi.sort_values('distancia').head(20)

In [ ]:
ruta = eda.mostrar_pares(casi[casi['entre_particiones']].sort_values('distancia'), RAIZ, n=6)
display(Mostrar(str(ruta))) if ruta else print('No hay pares entre particiones para mostrar')

## 7. Decisión del equipo: qué excluir
El conjunto de **prueba nunca se modifica**. La función retira de entrenamiento/validación las imágenes idénticas a
otra de una partición con más prioridad (prueba > validación > entrenamiento).

- `incluir_casi=False`: solo se retiran duplicados exactos.
- `incluir_casi=True`: también los casi duplicados entre particiones (después de revisarlos arriba).

Registren la decisión y el motivo en `docs/02_protocolo.md` y en la bitácora.

In [ ]:
INCLUIR_CASI = False   # <- decisión del equipo

excluidos = eda.generar_exclusiones(inventario, casi, incluir_casi=INCLUIR_CASI)
excluidos.to_csv('resultados/excluidos.csv', index=False)
print('Imágenes que se retiran:', len(excluidos))
excluidos.head(20)

In [ ]:
# Comprobación: cómo quedan las particiones que usarán TODOS los experimentos
from src.dataset import cargar_particiones
partes = cargar_particiones()
{nombre: len(tabla) for nombre, tabla in partes.items()}